# Generate the IEEE Access "Highlighted PDF" (latexdiff)

This notebook marks every change between your **originally-submitted** manuscript and your
**revised** manuscript, producing a PDF where additions are underlined/blue and deletions
are struck-through/red. Upload that PDF to the portal as the **"Highlighted PDF."**

**What you need to upload (Step 2):** a single `.zip` containing the LaTeX project, i.e.:
`NC_IEEE_Access.tex` (revised), `old_NC_IEEE_Access_submitted_Mar30.tex` (original),
`nc_references.bib`, `ieeeaccess.cls`, `IEEEtran.bst`, `rupa.jpg`, and the `figures/` folder.

Run the cells in order (Runtime ▸ Run all).

## Step 1 — Install latexdiff + LaTeX  (takes a few minutes)

In [ ]:
import subprocess, sys
print("Installing latexdiff and TeX Live packages...")
subprocess.run(["apt-get","update","-qq"], check=False)
pkgs=["latexdiff","texlive-latex-base","texlive-latex-extra","texlive-publishers",
      "texlive-science","texlive-fonts-recommended","texlive-fonts-extra","latexmk"]
subprocess.run(["apt-get","install","-y","-qq"]+pkgs, check=False)
ok = subprocess.run(["which","latexdiff"], capture_output=True, text=True).stdout.strip()
print("latexdiff:", ok or "NOT FOUND (re-run this cell)")
print("pdflatex :", subprocess.run(["which","pdflatex"],capture_output=True,text=True).stdout.strip())

## Step 2 — Upload your project `.zip`
Run the cell, click **Choose Files**, and pick the `.zip` described above.
(Alternative: comment this out and mount Drive instead — see the last cell.)

In [ ]:
import os, zipfile, glob
from google.colab import files
up = files.upload()                      # pick your .zip
zname = next(k for k in up if k.lower().endswith(".zip"))
work = "/content/diffbuild"
os.makedirs(work, exist_ok=True)
with zipfile.ZipFile(zname) as z: z.extractall(work)

# If the zip had a top-level folder, descend into the dir that holds the revised .tex
hits = glob.glob(os.path.join(work,"**","NC_IEEE_Access.tex"), recursive=True)
assert hits, "NC_IEEE_Access.tex not found in the zip"
JOBDIR = os.path.dirname(hits[0])
os.chdir(JOBDIR)
print("Working dir:", JOBDIR)
print("Files present:", sorted(os.listdir(".")))

## Step 3 — Set the two filenames and run latexdiff
Edit `OLD_TEX` / `NEW_TEX` if your filenames differ. The options below are chosen for
robustness on this manuscript (safe macros, no figure-frame markup, citations left intact).

In [ ]:
import subprocess, os
OLD_TEX = "old_NC_IEEE_Access_submitted_Mar30.tex"   # originally submitted (reviewed) version
NEW_TEX = "NC_IEEE_Access.tex"                        # current revised version
DIFF    = "NC_highlighted.tex"

assert os.path.exists(OLD_TEX), f"missing {OLD_TEX} (put it in the zip)"
assert os.path.exists(NEW_TEX), f"missing {NEW_TEX}"

cmd = ["latexdiff",
       "--append-safecmd=featnorm,fnthresh,tNC",   # custom macros: treat as safe
       "--graphics-markup=none",                    # don't frame figures (avoids missing-file errors)
       "--disable-citation-markup",                 # keep \cite intact (old keys were renamed)
       "--math-markup=whole",                       # mark whole equations, not token-by-token
       OLD_TEX, NEW_TEX]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
open(DIFF,"w").write(res.stdout)
print("stderr (first lines):\n", "\n".join(res.stderr.splitlines()[:8]))
print(f"\nWrote {DIFF}: {os.path.getsize(DIFF)} bytes" if os.path.getsize(DIFF)>0 else "EMPTY OUTPUT - see stderr above")

## Step 4 — Compile the highlighted PDF

In [ ]:
import subprocess, os
base = "NC_highlighted"
def run(c): 
    r=subprocess.run(c, capture_output=True, text=True); return r
run(["pdflatex","-interaction=nonstopmode",base+".tex"])
run(["bibtex",base])
run(["pdflatex","-interaction=nonstopmode",base+".tex"])
r=run(["pdflatex","-interaction=nonstopmode",base+".tex"])
errs=[l for l in r.stdout.splitlines() if l.startswith("!")]
print("PDF created:", os.path.exists(base+".pdf"), "| size:",
      os.path.getsize(base+".pdf") if os.path.exists(base+".pdf") else 0, "bytes")
print("LaTeX errors:", errs[:10] if errs else "none")
print("\nIf errors mention an UNDEFINED macro, add it to --append-safecmd in Step 3 and re-run Steps 3-4.")

## Step 5 — Download the Highlighted PDF

In [ ]:
from google.colab import files
import os
if os.path.exists("NC_highlighted.pdf") and os.path.getsize("NC_highlighted.pdf")>0:
    files.download("NC_highlighted.pdf")
else:
    print("No PDF yet - fix the errors reported in Step 4, then re-run Step 4.")

## Troubleshooting
- **`latexdiff: command not found`** — re-run Step 1.
- **Compile errors on a custom command** (e.g. `\PARstart`, `\featnorm`) — add the command name to
  `--append-safecmd=...` in Step 3 (comma-separated, no backslash) and re-run Steps 3–4.
- **Missing figure / `rupa.jpg`** — make sure the `figures/` folder and `rupa.jpg` are inside the zip.
- **`ieeeaccess.cls not found`** — it is not in TeX Live; it must be in the zip (it is part of your submission folder).
- **Too much markup / unreadable tables** — re-run Step 3 adding `--exclude-textcmd="tabular"` or
  change `--math-markup=whole` to `--math-markup=off`.

### Alternative input: Google Drive instead of a zip
Replace Step 2 with:
```python
from google.colab import drive; drive.mount('/content/drive')
import os; JOBDIR='/content/drive/MyDrive/NC_highlight'   # folder holding all the files
os.chdir(JOBDIR); print(sorted(os.listdir('.')))
```